In [2]:
import os
print(os.getcwd())

/Users/anitanam/optical_materials_ml/notebooks


In [3]:
ri_structure = pd.read_parquet(
    "data/processed/ri_structure_dataset.parquet"
)

ri_structure.shape

(32341, 142)

In [4]:
import os

for root, dirs, files in os.walk("."):
    for f in files:
        if "ri_structure" in f:
            print(os.path.join(root, f))

./data/processed/ri_structure_dataset.parquet


In [10]:
import pandas as pd

ri_structure = pd.read_parquet(
    "data/processed/ri_structure_dataset.parquet"
)

ri_structure.shape

(32341, 142)

In [11]:
ri_structure.columns

Index(['reduced_formula', 'refractive_index', 'optb88vdw_bandgap',
       'formation_energy_peratom', 'ehull', 'epsx', 'epsy', 'epsz',
       'MagpieData minimum Number', 'MagpieData maximum Number',
       ...
       'MagpieData avg_dev GSmagmom', 'MagpieData mode GSmagmom',
       'MagpieData minimum SpaceGroupNumber',
       'MagpieData maximum SpaceGroupNumber',
       'MagpieData range SpaceGroupNumber', 'MagpieData mean SpaceGroupNumber',
       'MagpieData avg_dev SpaceGroupNumber',
       'MagpieData mode SpaceGroupNumber', 'jid', 'atoms'],
      dtype='object', length=142)

In [13]:
ri_structure[["jid", "reduced_formula", "refractive_index"]].head()

,jid,reduced_formula,refractive_index
0,JVASP-90856,TiCuSiAs,8.296590
1,JVASP-86097,DyB6,11.873256
2,JVASP-64906,Be2OsRu,14.027763
3,JVASP-64664,Ba4NaBi,10.828656
4,JVASP-86726,LuNi4Sn,15.273380


In [14]:
ri_structure = pd.read_parquet("data/processed/ri_structure_dataset.parquet")

ri_structure[["jid", "reduced_formula", "refractive_index"]].head()

,jid,reduced_formula,refractive_index
0,JVASP-90856,TiCuSiAs,8.296590
1,JVASP-86097,DyB6,11.873256
2,JVASP-64906,Be2OsRu,14.027763
3,JVASP-64664,Ba4NaBi,10.828656
4,JVASP-86726,LuNi4Sn,15.273380


In [16]:
type(ri_structure.iloc[0]["atoms"])

dict

In [17]:
ri_structure.iloc[0]["atoms"]

{'lattice_mat': array([array([ 3.56693322,  0.        , -0.        ]),
        array([ 0.        ,  3.56693322, -0.        ]),
        array([-0.        , -0.        ,  9.39707545])], dtype=object),
 'coords': array([array([2.6751975 , 2.6751975 , 7.37610175]),
        array([0.8917325 , 0.8917325 , 2.02097825]),
        array([0.8917325, 2.6751975, 4.69854  ]),
        array([2.6751975, 0.8917325, 4.69854  ]),
        array([0.8917325, 2.6751975, 0.       ]),
        array([2.6751975, 0.8917325, 0.       ]),
        array([2.6751975 , 2.6751975 , 2.88947956]),
        array([0.8917325 , 0.8917325 , 6.50760044])], dtype=object),
 'elements': array(['Ti', 'Ti', 'Cu', 'Cu', 'Si', 'Si', 'As', 'As'], dtype=object),
 'abc': array([3.56693, 3.56693, 9.39708]),
 'angles': array([90., 90., 90.]),
 'cartesian': True,
 'props': array(['', '', '', '', '', '', '', ''], dtype=object)}

In [18]:
type(ri_structure.iloc[0]["atoms"]["lattice_mat"])

numpy.ndarray

In [19]:
import numpy as np
from jarvis.core.atoms import Atoms

def clean_atoms_dict(ad):
    return {
        "lattice_mat": np.asarray(ad["lattice_mat"], dtype=float).tolist(),
        "coords": np.asarray(ad["coords"], dtype=float).tolist(),
        "elements": [str(x) for x in np.asarray(ad["elements"]).tolist()],
        "abc": np.asarray(ad["abc"], dtype=float).tolist(),
        "angles": np.asarray(ad["angles"], dtype=float).tolist(),
        "cartesian": bool(np.asarray(ad["cartesian"]).item()),
        "props": [str(x) for x in np.asarray(ad["props"]).tolist()],
    }

raw_atoms = ri_structure.iloc[0]["atoms"]
clean_atoms = clean_atoms_dict(raw_atoms)

atoms = Atoms.from_dict(clean_atoms)
atoms

ValueError: setting an array element with a sequence.

In [20]:
from jarvis.core.atoms import Atoms

# Replace the serialized atoms column with fresh raw JARVIS structure dicts
ri_structure["atoms_raw"] = ri_structure["jid"].map(structure_lookup)

# Test one row
raw_atoms = ri_structure.iloc[0]["atoms_raw"]
atoms = Atoms.from_dict(raw_atoms)

atoms

NameError: name 'structure_lookup' is not defined

In [21]:
from jarvis.db.figshare import data

dft3d = data("dft_3d")

structure_lookup = {
    x["jid"]: x["atoms"]
    for x in dft3d
}

print(len(structure_lookup))

Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.
93902


In [22]:
from jarvis.core.atoms import Atoms

ri_structure["atoms_raw"] = (
    ri_structure["jid"]
    .map(structure_lookup)
)

raw_atoms = ri_structure.iloc[0]["atoms_raw"]

atoms = Atoms.from_dict(raw_atoms)

print("Atoms object created successfully")
print("Number of atoms:", atoms.num_atoms)
print("Composition:", atoms.composition)

Atoms object created successfully
Number of atoms: 8
Composition: OrderedDict({'Ti': 2, 'Cu': 2, 'Si': 2, 'As': 2})


In [23]:
ri_structure["atoms_raw"].notna().mean()

np.float64(1.0)

In [24]:
ri_structure = ri_structure.drop(
    columns=["atoms"],
    errors="ignore"
)

ri_structure = ri_structure.rename(
    columns={"atoms_raw": "atoms"}
)

ri_structure.to_parquet(
    "data/processed/ri_structure_dataset_clean.parquet",
    index=False
)

In [25]:
from jarvis.core.atoms import Atoms

ri_structure["atoms_raw"] = ri_structure["jid"].map(structure_lookup)

raw_atoms = ri_structure.iloc[0]["atoms_raw"]
atoms = Atoms.from_dict(raw_atoms)

print("Atoms object created successfully")
print("Number of atoms:", atoms.num_atoms)
print("Composition:", atoms.composition)

Atoms object created successfully
Number of atoms: 8
Composition: OrderedDict({'Ti': 2, 'Cu': 2, 'Si': 2, 'As': 2})


In [26]:
ri_structure = ri_structure.drop(columns=["atoms"], errors="ignore")
ri_structure = ri_structure.rename(columns={"atoms_raw": "atoms"})

ri_structure.to_parquet(
    "data/processed/ri_structure_dataset_clean.parquet",
    index=False
)

In [27]:
import os
os.path.exists("data/processed/ri_structure_dataset_clean.parquet")

True

In [28]:
print(ri_structure.shape)

sample_atoms = ri_structure.iloc[0]["atoms"]
print(type(sample_atoms))

(32341, 142)
<class 'dict'>
